# A Day in the Life of a Fleet — Telemetry, Meet Intelligence

**Every delivery operator already has this data.** Tens of thousands of
GPS pings every hour, bouncing off buildings, drifting through parking
lots, disappearing into tunnels. It sits in a data lake nobody opens.

This notebook is what happens when you point Wherobots at it. We take one
day of real telemetry — **~50 trucks, ~100 trips, hundreds of thousands
of pings** — and in one notebook turn it into the dashboard the
dispatcher, the finance team, and the customer-success rep all want:

- **Clean trajectories** — every trace snapped to the actual road driven,
  with noise thrown out.
- **Zone transitions highlighted** — the moment a truck enters a
  customer's geofence is the moment the billable clock starts.
- **Per-driver stories** — who spent the most time where, who kept
  their schedule, who got lost in traffic.
- **One map** — everything overlaid so the operations lead can read the
  entire day in ten seconds.

Data: the [Vehicle Energy Dataset (VED)](https://github.com/gsoh/VED) —
real GPS traces across Ann Arbor, Michigan.

## 1. Setup

Sedona for SQL-over-spatial, Wherobots Map Matcher for the snap-to-road
step, SedonaKepler for the map.

In [ ]:
from sedona.spark import *
from wherobots import matcher
from shapely.geometry import LineString
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, expr, udf, collect_list, struct, row_number, lit
)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

## 2. The Raw Telemetry

Load a real week of GPS pings from S3, filter to a single representative
day, and give ourselves a quick sense of scale.

In [ ]:
GPS_S3_PATH = "s3://wherobots-examples/data/VED_171101_week.csv"

raw_gps_df = sedona.read.csv(GPS_S3_PATH, header=True, inferSchema=True) \
    .selectExpr(
        "VehId            AS vehicle_id",
        "Trip             AS trip_id",
        "`Timestamp(ms)`  AS ts_ms",
        "`Latitude[deg]`  AS lat",
        "`Longitude[deg]` AS lon",
    )

stats = raw_gps_df.agg(
    expr("COUNT(*)                          AS pings"),
    expr("COUNT(DISTINCT vehicle_id)        AS vehicles"),
    expr("COUNT(DISTINCT trip_id)           AS trips"),
).first()

print("══════════════════════════════════════════════")
print(f"  {stats.pings:>10,}  GPS pings in scope")
print(f"  {stats.vehicles:>10,}  vehicles")
print(f"  {stats.trips:>10,}  unique trips")
print("══════════════════════════════════════════════")

## 3. Trips — From Pings to Journeys

Every `(vehicle_id, trip_id)` pair becomes one LineString — the raw,
noisy path the truck actually traced. We cap the demo at 100 trips so
the map stays readable; in production this scales across tens of
thousands of concurrent vehicles.

In [ ]:
def rows_to_linestring(rows):
    srt = sorted(rows, key=lambda x: x["ts_ms"])
    return LineString([(r["lon"], r["lat"]) for r in srt])

linestring_udf = udf(rows_to_linestring, GeometryType())

TRIP_LIMIT = 100

trips_df = (
    raw_gps_df
    .groupBy("vehicle_id", "trip_id")
    .agg(collect_list(struct("ts_ms", "lat", "lon")).alias("coords"))
    .withColumn("geometry", linestring_udf("coords"))
)

ids_window = Window.partitionBy(lit(1)).orderBy("vehicle_id", "trip_id")
trips_df = (
    trips_df
    .withColumn("ids", row_number().over(ids_window) - 1)
    .filter(col("ids") < TRIP_LIMIT)
    .select("ids", "vehicle_id", "trip_id", "coords", "geometry")
    .cache()
)

print(f"Trips loaded for the day: {trips_df.count()}")

## 4. Snap to the Road — Map Matching

Raw GPS lives on an urban canyon, bouncing a few meters off buildings.
Before it's useful, it needs to be **snapped** to the actual road
driven. Wherobots' Map Matcher takes an OSM road graph and every
observed trip and returns the likeliest sequence of roads.

One call. Everything parallelizes.

In [ ]:
ROADS_S3_PATH = "s3://wherobots-examples/data/osm_AnnArbor_large.xml"

roads_df = matcher.load_osm(ROADS_S3_PATH, "[car]").cache()
print(f"OSM road segments loaded: {roads_df.count():,}")

sedona.conf.set("wherobots.tools.mm.maxdist",     "100")
sedona.conf.set("wherobots.tools.mm.maxdistinit", "100")
sedona.conf.set("wherobots.tools.mm.obsnoise",    "40")

matched_df = matcher.match(roads_df, trips_df, "geometry", "geometry").cache()
print(f"Trips successfully map-matched: {matched_df.count()}")

## 5. Service Zones — the Customer Footprint

This fleet's contracts cover five geofenced areas — two retail clusters,
two university campuses, and an industrial distribution district.
Every billable minute starts when a truck crosses one of these
boundaries.

In [ ]:
# (zone_name, category, min_lon, min_lat, max_lon, max_lat, rate_per_hour_usd)
zones = [
    ("Downtown Ann Arbor",  "Retail",        -83.755, 42.276, -83.735, 42.286, 85.0),
    ("UM Central Campus",   "Institutional", -83.746, 42.264, -83.730, 42.279, 65.0),
    ("UM North / Tech",     "Institutional", -83.725, 42.290, -83.700, 42.308, 65.0),
    ("South Industrial",    "Industrial",    -83.758, 42.232, -83.720, 42.256, 45.0),
    ("East Commercial",     "Retail",        -83.702, 42.262, -83.678, 42.288, 75.0),
]

zone_schema = StructType([
    StructField("zone_name",         StringType()),
    StructField("category",          StringType()),
    StructField("min_lon",           DoubleType()),
    StructField("min_lat",           DoubleType()),
    StructField("max_lon",           DoubleType()),
    StructField("max_lat",           DoubleType()),
    StructField("rate_per_hour_usd", DoubleType()),
])

zones_df = sedona.createDataFrame(zones, zone_schema) \
    .withColumn("geometry",
                expr("ST_MakeEnvelope(min_lon, min_lat, max_lon, max_lat, 4326)"))
zones_df.createOrReplaceTempView("zones")
zones_df.select("zone_name", "category", "rate_per_hour_usd").show(truncate=False)

## 6. Zone Transitions — the Heartbeat of the Day

Every ping gets tagged with the zone it's in (or `IN_TRANSIT`). A
`LAG()` window detects the exact ping where the zone changes — that's
the moment a delivery starts or ends. Group consecutive same-zone pings
together and you get **visits**.

In [ ]:
trip_keys_df = trips_df.select("vehicle_id", "trip_id")
pings_in_scope = raw_gps_df.join(trip_keys_df, ["vehicle_id", "trip_id"])
pings_in_scope.createOrReplaceTempView("pings")

sedona.sql("""
    CREATE OR REPLACE TEMP VIEW visit_pings AS
    WITH tagged AS (
        SELECT p.vehicle_id, p.trip_id, p.ts_ms, p.lat, p.lon,
               COALESCE(z.zone_name,         'IN_TRANSIT') AS zone_name,
               COALESCE(z.category,          '')           AS category,
               COALESCE(z.rate_per_hour_usd, 0.0)          AS rate_per_hour_usd
        FROM pings p
        LEFT JOIN zones z
          ON ST_Contains(z.geometry, ST_Point(p.lon, p.lat))
    ),
    with_prev AS (
        SELECT *,
               LAG(zone_name) OVER (
                 PARTITION BY vehicle_id, trip_id ORDER BY ts_ms
               ) AS prev_zone
        FROM tagged
    )
    SELECT vehicle_id, trip_id, ts_ms, lat, lon,
           zone_name, category, rate_per_hour_usd,
           SUM(CASE WHEN prev_zone IS NULL OR zone_name != prev_zone
                    THEN 1 ELSE 0 END)
             OVER (PARTITION BY vehicle_id, trip_id ORDER BY ts_ms)
             AS visit_id
    FROM with_prev
""")

visits_df = sedona.sql("""
    SELECT
        vehicle_id, trip_id, visit_id,
        zone_name, category, rate_per_hour_usd,
        MIN(ts_ms)                                  AS enter_ts,
        MAX(ts_ms)                                  AS exit_ts,
        COUNT(*)                                    AS ping_count,
        ROUND((MAX(ts_ms) - MIN(ts_ms)) / 1000.0, 1) AS dwell_seconds
    FROM visit_pings
    WHERE zone_name != 'IN_TRANSIT'
    GROUP BY vehicle_id, trip_id, visit_id,
             zone_name, category, rate_per_hour_usd
    HAVING COUNT(*) >= 2
""").cache()
visits_df.createOrReplaceTempView("zone_visits")

print(f"Zone visits detected across the day: {visits_df.count()}")

## 7. The Dashboard — One Glance Per Zone

In [ ]:
dashboard = sedona.sql("""
    SELECT
        zone_name,
        category,
        COUNT(*)                                   AS visits,
        COUNT(DISTINCT vehicle_id)                 AS trucks,
        ROUND(SUM(dwell_seconds) / 60.0, 1)        AS dwell_minutes,
        ROUND(AVG(dwell_seconds), 1)               AS avg_visit_seconds,
        ROUND(
            SUM(dwell_seconds) / 3600.0 * rate_per_hour_usd, 2
        ) AS billable_usd
    FROM zone_visits
    GROUP BY zone_name, category, rate_per_hour_usd
    ORDER BY dwell_minutes DESC
""").cache()
dashboard.show(truncate=False)

totals = sedona.sql("""
    SELECT
        COUNT(*)                              AS total_visits,
        COUNT(DISTINCT vehicle_id)            AS active_trucks,
        ROUND(SUM(dwell_seconds) / 3600.0, 1) AS total_dwell_hours,
        ROUND(
            SUM(dwell_seconds) / 3600.0 * rate_per_hour_usd, 0
        ) AS total_billable_usd
    FROM zone_visits
""").first()

print("\n══════════════════════════════════════════════")
print(f"  {totals.total_visits:>6,}  zone visits")
print(f"  {totals.active_trucks:>6,}  active trucks")
print(f"  {totals.total_dwell_hours:>6.1f}  billable hours on-site")
print(f"  ${totals.total_billable_usd:>8,.0f}  in on-site billable time")
print("══════════════════════════════════════════════")

## 8. Featured Drivers — Three Stories

Pull the three trucks that racked up the most zone visits and narrate
their day — exactly what a dispatcher reviews after shift-end.

In [ ]:
top_drivers_df = sedona.sql("""
    SELECT
        vehicle_id,
        COUNT(*)                                AS visits,
        COUNT(DISTINCT zone_name)               AS unique_zones,
        ROUND(SUM(dwell_seconds) / 60.0, 1)     AS on_site_minutes
    FROM zone_visits
    GROUP BY vehicle_id
    ORDER BY visits DESC
    LIMIT 3
""").toPandas()

for _, drv in top_drivers_df.iterrows():
    print("──────────────────────────────────────────────")
    print(f"  Truck #{int(drv.vehicle_id):<4}   "
          f"{int(drv.visits)} visits   "
          f"{int(drv.unique_zones)} unique zones   "
          f"{drv.on_site_minutes:.1f} on-site minutes")
    print("──────────────────────────────────────────────")
    sedona.sql(f"""
        SELECT
            zone_name,
            COUNT(*)                              AS stops,
            ROUND(SUM(dwell_seconds) / 60.0, 1)   AS minutes,
            ROUND(
                SUM(dwell_seconds) / 3600.0 * rate_per_hour_usd, 2
            ) AS billable_usd
        FROM zone_visits
        WHERE vehicle_id = {int(drv.vehicle_id)}
        GROUP BY zone_name, rate_per_hour_usd
        ORDER BY minutes DESC
    """).show(truncate=False)

## 9. Zone Entry Markers — Where the Clock Started

Pull the first ping of each visit — these are the physical points where
the truck crossed the geofence. Each one is a billable event.

In [ ]:
entry_points_df = sedona.sql("""
    WITH ranked AS (
        SELECT v.*,
               ROW_NUMBER() OVER (
                 PARTITION BY vehicle_id, trip_id, visit_id ORDER BY ts_ms
               ) AS rn
        FROM visit_pings v
        WHERE zone_name != 'IN_TRANSIT'
    )
    SELECT
        vehicle_id, trip_id, zone_name, category, ts_ms,
        ST_Point(lon, lat) AS geometry
    FROM ranked
    WHERE rn = 1
""").cache()
print(f"Zone-entry events: {entry_points_df.count()}")

## 10. The Full Picture — One Map, Four Layers

Zones in purple, raw observed tracks in red, road-snapped matched routes
in green, zone-entry events as yellow dots. The whole operational day
in a single view.

In [ ]:
map_config = {
    "version": "v1",
    "config": {
        "mapState": {
            "latitude": 42.2749,
            "longitude": -83.723,
            "zoom": 11.8,
            "pitch": 0,
            "bearing": 0,
        },
        "mapStyle": {
            "styleType": "dark-matter",
            "visibleLayerGroups": {
                "label": True, "road": True, "building": True,
                "water": True, "land": True, "border": False,
            },
        },
    },
}

viz = SedonaKepler.create_map(config=map_config)
SedonaKepler.add_df(viz,
    zones_df.select("zone_name", "category",
                    "rate_per_hour_usd", "geometry"),
    name="Service Zones")
SedonaKepler.add_df(viz,
    trips_df.select("vehicle_id", "trip_id", "geometry"),
    name="Observed GPS (raw)")
SedonaKepler.add_df(viz,
    matched_df.selectExpr("ids", "matched_points AS geometry"),
    name="Matched Routes")
SedonaKepler.add_df(viz,
    entry_points_df.select("vehicle_id", "zone_name",
                           "category", "geometry"),
    name="Zone Entries")
viz

---

## Why this matters

The same data lake the operator already owns now feeds:

| Audience | What they see |
|---|---|
| **Dispatch** | Live map of trucks-in-zones, instant detour flags, route replays |
| **Finance** | Billable on-site hours per customer, per zone, per day — no manual timesheets |
| **Customer Success** | Per-customer SLA receipts: exact enter/exit times, proof of delivery |
| **Ops / Coaching** | Driver-level dwell patterns, problem intersections, detour analysis |
| **Sales** | "Here's your service footprint last month" — a visual the prospect can't ignore |

## What you just saw

- **One notebook.** Raw GPS → snapped tracks → zone-tagged → billable
  visits → executive dashboard → single map.
- **Two Wherobots calls.** `matcher.match(...)` for map matching, SQL
  `LAG()` window for zone transitions. That's it.
- **Every layer is a DataFrame.** Same code scales from 50 trucks to
  50,000 on a bigger runtime — no pipeline rewrite.